# 06 - 消融实验评测（论文核心）

**ClawTeam v3.1 真协作 Harness 版**

## 实验设计

### 主实验：4 组对比
| 编号 | 配置 | 验证 |
|------|------|------|
| E1 | 单 Agent (DeepSeek) | 基线 |
| E2 | 4 角色 Round 1 独立（无辩论 / 无 LoRA） | 角色分工本身有用？|
| E3 | 4 角色 + Round 2 辩论（无 LoRA） | Harness 辩论有用？ |
| E4 | E3 + 外科/内科 LoRA + Guardian | 完整方案 |

### 子消融
1. 训练 vs 不训练（外科 / 内科 LoRA vs API）
2. 复杂度评估方法对比（LLM / BERT / 关键词）
3. Round 数 acc vs cost 曲线

### 评测集
- MedQA 肿瘤子集
- MedBullets 肿瘤子集（如有）
- 自建 Tumor Board 案例集（手工，~30 题）

## Step 0: 环境准备

In [ ]:
import os, sys, asyncio, json, time
from pathlib import Path
from collections import defaultdict

# 让 notebook 能 import backend 模块
NOTEBOOK_DIR = Path.cwd()
EVAL_DIR = NOTEBOOK_DIR.parent
BACKEND_DIR = EVAL_DIR.parent
sys.path.insert(0, str(BACKEND_DIR))

os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')

DATA_DIR = EVAL_DIR / 'datasets' / 'data'
RESULTS_DIR = EVAL_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: 加载评测集（限制规模 100-200 题）

In [ ]:
# 加载肿瘤评测集
TEST_LIMIT = 100  # 调整：每个数据集最多评测 N 题（控制 API 成本）

def load_eval_set():
    test_cases = []

    # MedQA test 集肿瘤子集
    medqa_test = DATA_DIR / 'oncology' / 'medqa_test_oncology.jsonl'
    if medqa_test.exists():
        with open(medqa_test, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    item = json.loads(line)
                    options = item.get('options', [])
                    if isinstance(options, list) and options:
                        opt_text = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(options))
                        question_text = f"{item['question']}\n\n{opt_text}"
                    else:
                        question_text = item['question']
                    test_cases.append({
                        'source': 'MedQA',
                        'question': question_text,
                        'answer': item.get('answer_text', ''),
                        'answer_idx': item.get('answer_idx'),
                        'options': options,
                    })
                    if len(test_cases) >= TEST_LIMIT:
                        break

    # 自建 Tumor Board 案例集（如有）
    tumor_board = DATA_DIR / 'tumor_board_cases.jsonl'
    if tumor_board.exists():
        with open(tumor_board, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    item = json.loads(line)
                    test_cases.append({
                        'source': 'TumorBoard',
                        'question': item.get('case', ''),
                        'answer': item.get('expected_answer', ''),
                    })

    return test_cases

test_cases = load_eval_set()
print(f'评测集大小: {len(test_cases)}')
if test_cases:
    print(f'\n第一条样例:\n{json.dumps(test_cases[0], ensure_ascii=False, indent=2)[:300]}')

## Step 2: 准备评估器 — 4 种实验配置

In [ ]:
from graph.coordinator import Coordinator
from graph.complexity_assessor import CaseComplexity
from config import get_settings
from graph.llm import build_llm_config_from_settings, get_llm

coordinator = Coordinator(BACKEND_DIR)

# E1: 单 agent
async def run_e1_single_agent(case):
    settings = get_settings()
    llm = get_llm(build_llm_config_from_settings(settings, temperature=0.3, streaming=False))
    response = await llm.ainvoke([
        {'role': 'system', 'content': '你是医疗助手，回答肿瘤相关问题。'},
        {'role': 'user', 'content': case},
    ])
    return getattr(response, 'content', '').strip()

# E2: 4 角色独立（无辩论）
async def run_e2_no_debate(case):
    session = await coordinator.consult(
        case=case,
        force_complexity=CaseComplexity.MODERATE,  # 强制中等 → 跳过 Round 2
    )
    return session.final_decision

# E3: 4 角色 + Round 2 辩论（无 LoRA）
async def run_e3_with_debate(case):
    # 先确保 LoRA 关闭
    os.environ['USE_LORA_SURGEON'] = 'false'
    os.environ['USE_LORA_MEDICAL_ONCOLOGIST'] = 'false'
    session = await coordinator.consult(
        case=case,
        force_complexity=CaseComplexity.COMPLEX,  # 强制复杂 → 走 Round 2
    )
    return session.final_decision

# E4: E3 + LoRA（外科 + 内科）
async def run_e4_full(case):
    os.environ['USE_LORA_SURGEON'] = 'true'
    os.environ['LORA_SURGEON_BASE'] = 'Qwen/Qwen3-4B-Instruct'
    os.environ['LORA_SURGEON_PATH'] = 'eval/models/surgeon_qwen3_lora'
    os.environ['USE_LORA_MEDICAL_ONCOLOGIST'] = 'true'
    os.environ['LORA_MEDICAL_ONCOLOGIST_BASE'] = 'Qwen/Qwen3-4B-Instruct'
    os.environ['LORA_MEDICAL_ONCOLOGIST_PATH'] = 'eval/models/oncologist_qwen3_lora'
    session = await coordinator.consult(
        case=case,
        force_complexity=CaseComplexity.COMPLEX,
    )
    return session.final_decision, session

print('实验配置已加载')

## Step 3: 评测准确率（多选题用关键词匹配）

In [ ]:
def evaluate_mcq(predicted: str, expected: str, options: list = None) -> bool:
    """评估多选题是否答对（简单关键词匹配 + 选项字母）。"""
    if not predicted or not expected:
        return False

    predicted_lower = predicted.lower().strip()
    expected_lower = expected.lower().strip()

    # 直接命中
    if expected_lower in predicted_lower:
        return True

    # 选项字母（如 'A.' 'B、'）
    if options:
        for i, opt in enumerate(options):
            if str(opt).lower().strip() == expected_lower:
                letter = chr(65 + i)
                if letter in predicted or f'{letter}.' in predicted or f'{letter}、' in predicted:
                    return True

    return False


async def evaluate_method(method_name, method_fn, cases):
    """对每个 case 运行 method 并评估。"""
    correct = 0
    latencies = []
    sessions = []
    for i, case in enumerate(cases):
        start = time.monotonic()
        try:
            result = await method_fn(case['question'])
            if isinstance(result, tuple):
                result, session = result
                sessions.append(session)
        except Exception as exc:
            print(f'  [{i+1}/{len(cases)}] {method_name} 失败: {exc}')
            result = ''
        elapsed = time.monotonic() - start
        latencies.append(elapsed)

        if evaluate_mcq(result, case['answer'], case.get('options')):
            correct += 1

        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{len(cases)}] {method_name}: 当前准确率 {correct/(i+1):.3f}')

    return {
        'method': method_name,
        'total': len(cases),
        'correct': correct,
        'accuracy': correct / len(cases) if cases else 0,
        'avg_latency_s': sum(latencies) / len(latencies) if latencies else 0,
        'sessions': sessions,
    }

## Step 4: 跑主实验（4 组）

In [ ]:
# 主实验 4 组（请按顺序跑，避免互相影响）
results = {}

print('=== E1: 单 Agent ===')
results['E1'] = await evaluate_method('E1_single', run_e1_single_agent, test_cases)
print(f'E1 准确率: {results["E1"]["accuracy"]:.3f}\n')

In [ ]:
print('=== E2: 4 角色独立（无辩论 / 无 LoRA）===')
results['E2'] = await evaluate_method('E2_no_debate', run_e2_no_debate, test_cases)
print(f'E2 准确率: {results["E2"]["accuracy"]:.3f}\n')

In [ ]:
print('=== E3: 4 角色 + Round 2 辩论（无 LoRA）===')
results['E3'] = await evaluate_method('E3_debate', run_e3_with_debate, test_cases)
print(f'E3 准确率: {results["E3"]["accuracy"]:.3f}\n')

In [ ]:
print('=== E4: 完整方案（辩论 + LoRA）===')
results['E4'] = await evaluate_method('E4_full', run_e4_full, test_cases)
print(f'E4 准确率: {results["E4"]["accuracy"]:.3f}')

# 计算 E4 的协作有效性指标
if results['E4']['sessions']:
    rev_rates = [s.revision_rate for s in results['E4']['sessions']]
    print(f'平均修正率（Revision Rate）: {sum(rev_rates)/len(rev_rates):.3f}')

## Step 5: 主实验汇总表（论文 Table 1）

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        '实验': k,
        '配置': {
            'E1': '单 Agent (baseline)',
            'E2': '4 角色独立',
            'E3': '+ Round 2 辩论',
            'E4': '+ LoRA + Guardian',
        }[k],
        '准确率': f"{v['accuracy']:.3f}",
        '平均延迟(s)': f"{v['avg_latency_s']:.2f}",
        '提升 vs E1': f"{(v['accuracy'] - results['E1']['accuracy']) * 100:+.1f}%",
    }
    for k, v in results.items()
])
print('\n=== 主实验结果 ===')
print(summary.to_string(index=False))

# 保存
summary.to_csv(RESULTS_DIR / 'main_ablation.csv', index=False, encoding='utf-8-sig')
print(f'\n已保存到: {RESULTS_DIR / "main_ablation.csv"}')

## Step 6: 子消融 1 — 训练 vs 不训练（外科 / 内科）

In [ ]:
# 在 E3（无 LoRA）和 E4（有 LoRA）的基础上拆分对比
# 输出 Table 2：单角色训练前后
# 这部分需要单独跑：屏蔽其他 3 个角色，只看外科 / 内科表现

# (略，实际使用时按需展开)
print('子消融 1: 已通过 E3 vs E4 对比体现训练效果')

## Step 7: 子消融 2 — 复杂度评估方法对比

In [ ]:
from graph.complexity_assessor import assess_complexity

print('=== 复杂度评估方法对比（前 30 个样例） ===')
for method in ['llm', 'bert', 'keyword']:
    correct_routing = 0
    for case in test_cases[:30]:
        try:
            decision = await assess_complexity(case['question'], method=method)
            # 我们没有黄金路由标签，所以这里只统计分布
            correct_routing += 1
        except Exception as exc:
            print(f'  {method} 失败: {exc}')
    print(f'{method}: 处理成功 {correct_routing}/30')

## Step 8: 子消融 3 — Round 数 acc vs cost

In [ ]:
# 比较 Round 1（独立）vs Round 1+2（辩论）
# E2 = 1 Round, E3 = 2 Rounds
round_comp = pd.DataFrame([
    {
        'Round 数': '1（独立）',
        '准确率': f"{results['E2']['accuracy']:.3f}",
        '平均延迟(s)': f"{results['E2']['avg_latency_s']:.2f}",
        '相对成本': '1x',
    },
    {
        'Round 数': '2（独立 + 辩论）',
        '准确率': f"{results['E3']['accuracy']:.3f}",
        '平均延迟(s)': f"{results['E3']['avg_latency_s']:.2f}",
        '相对成本': f'{results["E3"]["avg_latency_s"] / results["E2"]["avg_latency_s"]:.1f}x',
    },
])
print('\n=== Round 数对比 ===')
print(round_comp.to_string(index=False))
round_comp.to_csv(RESULTS_DIR / 'round_comparison.csv', index=False, encoding='utf-8-sig')

## Step 9: 论文用图表（matplotlib）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 主实验柱状图
fig, ax = plt.subplots(figsize=(10, 6))
names = ['E1\n单 Agent', 'E2\n4 角色独立', 'E3\n+辩论', 'E4\n+LoRA']
accs = [results[k]['accuracy'] for k in ['E1', 'E2', 'E3', 'E4']]
colors = ['#888', '#5B9BD5', '#ED7D31', '#70AD47']
bars = ax.bar(names, accs, color=colors)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01, f'{acc:.3f}',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('准确率', fontsize=12)
ax.set_title('ClawTeam 主消融实验（Tumor Board 任务）', fontsize=14)
ax.set_ylim(0, max(accs) * 1.2)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'main_ablation.png', dpi=150)
plt.show()
print(f'\n图已保存: {RESULTS_DIR / "main_ablation.png"}')

## Step 10: 总结报告（论文用）

In [ ]:
report = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'eval_set_size': len(test_cases),
    'main_ablation': {k: {'accuracy': v['accuracy'], 'avg_latency_s': v['avg_latency_s']} for k, v in results.items()},
    'key_findings': {
        '角色分工提升 (E2 vs E1)': f"{(results['E2']['accuracy'] - results['E1']['accuracy']) * 100:+.1f}%",
        '辩论提升 (E3 vs E2)': f"{(results['E3']['accuracy'] - results['E2']['accuracy']) * 100:+.1f}%",
        '训练提升 (E4 vs E3)': f"{(results['E4']['accuracy'] - results['E3']['accuracy']) * 100:+.1f}%",
        '总提升 (E4 vs E1)': f"{(results['E4']['accuracy'] - results['E1']['accuracy']) * 100:+.1f}%",
    },
}

if results['E4']['sessions']:
    rev_rates = [s.revision_rate for s in results['E4']['sessions']]
    report['collaboration_metrics'] = {
        'avg_revision_rate': sum(rev_rates) / len(rev_rates),
        'note': 'Round 2 中明确给出修正的角色占比（论文核心指标，证明真协作）',
    }

with open(RESULTS_DIR / 'final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))